# Text Generation Metrics: BLEU, ROUGE & METEOR

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/text-generation-metrics)

We implement BLEU (modified n-gram precision + brevity penalty), ROUGE-N / ROUGE-L (recall-oriented n-gram / LCS overlap), and a simplified METEOR (precision/recall harmonic mean + fragmentation penalty) from scratch in plain Python, then score a few candidate/reference pairs — including one where a good paraphrase exposes why these metrics motivate LLM-as-a-judge for open-ended generation.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
from collections import Counter
import math

def tokenize(s):
    return s.lower().split()

reference = tokenize("the cat sat on the mat")
candidate = tokenize("the cat sat on mat")

print("reference:", reference)
print("candidate:", candidate)

## 1 — BLEU: modified n-gram precision + brevity penalty

Modified precision clips each candidate n-gram's count at how many times it appears in the reference, so repeating a frequent word can't inflate the score:

$$p_n = \frac{\sum_{\text{n-gram} \in C} \min(\text{count}_C, \text{count}_R)}{\sum_{\text{n-gram} \in C} \text{count}_C}$$

Then BLEU combines $p_1, \ldots, p_N$ with a geometric mean and multiplies by a brevity penalty that punishes candidates shorter than the reference.

In [ ]:
def get_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def modified_precision(candidate, reference, n):
    cand_ngrams = Counter(get_ngrams(candidate, n))
    ref_ngrams = Counter(get_ngrams(reference, n))
    if not cand_ngrams:
        return 0.0
    clipped = sum(min(count, ref_ngrams[ng]) for ng, count in cand_ngrams.items())
    total = sum(cand_ngrams.values())
    return clipped / total

def brevity_penalty(candidate, reference):
    c, r = len(candidate), len(reference)
    if c > r:
        return 1.0
    if c == 0:
        return 0.0
    return math.exp(1 - r / c)

def bleu(candidate, reference, max_n=4):
    precisions = [modified_precision(candidate, reference, n) for n in range(1, max_n + 1)]
    if min(precisions) == 0:
        geo_mean = 0.0
    else:
        geo_mean = math.exp(sum(math.log(p) for p in precisions) / len(precisions))
    return brevity_penalty(candidate, reference) * geo_mean

for n in range(1, 3):
    print(f"p_{n} =", round(modified_precision(candidate, reference, n), 4))
print("brevity penalty =", round(brevity_penalty(candidate, reference), 4))
print("BLEU-2 (n=1,2) =", round(bleu(candidate, reference, max_n=2), 4))

## 2 — ROUGE-N and ROUGE-L: recall-oriented overlap

ROUGE-N flips BLEU's ratio: same clipped-overlap numerator, but divided by the **reference's** n-gram count — *how much of the reference did the candidate reproduce?* ROUGE-L instead uses the Longest Common Subsequence, which rewards preserving order without demanding a contiguous n-gram match.

In [ ]:
def rouge_n_recall(candidate, reference, n):
    cand_ngrams = Counter(get_ngrams(candidate, n))
    ref_ngrams = Counter(get_ngrams(reference, n))
    if not ref_ngrams:
        return 0.0
    clipped = sum(min(count, cand_ngrams[ng]) for ng, count in ref_ngrams.items())
    total = sum(ref_ngrams.values())
    return clipped / total

def lcs_length(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]

def rouge_l(candidate, reference, beta=1.2):
    l = lcs_length(candidate, reference)
    if l == 0:
        return 0.0, 0.0, 0.0
    p_lcs = l / len(candidate)
    r_lcs = l / len(reference)
    f_lcs = ((1 + beta ** 2) * p_lcs * r_lcs) / (r_lcs + beta ** 2 * p_lcs)
    return p_lcs, r_lcs, f_lcs

print("ROUGE-1 recall =", round(rouge_n_recall(candidate, reference, 1), 4))
p_lcs, r_lcs, f_lcs = rouge_l(candidate, reference)
print(f"ROUGE-L: P={p_lcs:.4f}  R={r_lcs:.4f}  F={f_lcs:.4f}")

## 3 — METEOR: precision/recall harmonic mean + fragmentation penalty

A full METEOR implementation stages exact → stemmed → synonym matching. We implement the **exact-match stage only** (which is enough to reproduce the wiki page's worked example), plus the fragmentation penalty based on the number of contiguous matched chunks:

$$F_{\text{mean}} = \frac{10PR}{R + 9P}, \qquad \text{Penalty} = 0.5\Big(\frac{\#\text{chunks}}{\#\text{matches}}\Big)^3, \qquad \text{METEOR} = F_{\text{mean}}(1 - \text{Penalty})$$

In [ ]:
def align_exact(candidate, reference):
    """Greedy exact-match alignment: for each candidate position, match the
    earliest unused identical reference position. Returns list of (i, j) pairs."""
    used_ref = set()
    matches = []
    for i, tok in enumerate(candidate):
        for j, rtok in enumerate(reference):
            if j in used_ref:
                continue
            if tok == rtok:
                matches.append((i, j))
                used_ref.add(j)
                break
    return matches

def count_chunks(matches):
    """Count contiguous runs where both the candidate index and the reference
    index increase by exactly 1 from the previous match."""
    if not matches:
        return 0
    matches = sorted(matches)
    chunks = 1
    for (i0, j0), (i1, j1) in zip(matches, matches[1:]):
        if not (i1 == i0 + 1 and j1 == j0 + 1):
            chunks += 1
    return chunks

def meteor(candidate, reference):
    matches = align_exact(candidate, reference)
    m = len(matches)
    if m == 0:
        return 0.0
    p = m / len(candidate)
    r = m / len(reference)
    f_mean = (10 * p * r) / (r + 9 * p)
    chunks = count_chunks(matches)
    penalty = 0.5 * (chunks / m) ** 3
    return f_mean * (1 - penalty)

matches = align_exact(candidate, reference)
print("matched pairs (candidate_idx, reference_idx):", matches)
print("chunks:", count_chunks(matches))
print("METEOR =", round(meteor(candidate, reference), 4))

All three metrics agree the candidate is close but imperfect — matching the wiki page's hand-worked BLEU-2 ≈ 0.709, ROUGE-1 recall ≈ 0.833 / ROUGE-L F ≈ 0.909, and METEOR ≈ 0.820.

## 4 — The blind spot: a good paraphrase scores near zero

`candidate_paraphrase` says the same thing as the reference in different words, with almost no shared n-grams. A human (or an LLM judge) would call it correct; every overlap-based metric here punishes it almost as much as an unrelated sentence.

In [ ]:
candidate_paraphrase = tokenize("a feline was resting on the rug")
candidate_unrelated = tokenize("stock markets rallied on strong earnings")

for name, cand in [
    ("close paraphrase (dropped 'the')", candidate),
    ("good paraphrase, different words", candidate_paraphrase),
    ("unrelated sentence", candidate_unrelated),
]:
    b = bleu(cand, reference, max_n=2)
    r1 = rouge_n_recall(cand, reference, 1)
    _, _, rl_f = rouge_l(cand, reference)
    met = meteor(cand, reference)
    print(f"{name:35s}  BLEU-2={b:.3f}  ROUGE-1={r1:.3f}  ROUGE-L={rl_f:.3f}  METEOR={met:.3f}")

print()
print("The paraphrase scores almost as low as the unrelated sentence on every metric —")
print("surface overlap can't see that its *meaning* matches the reference. This is exactly")
print("the gap LLM-as-a-judge fills for open-ended generation.")

## ✏️ Your turn

**Task — generalize ROUGE-N to bigrams:** you've seen `rouge_n_recall` computed for unigrams (`n=1`). Reuse it to compute the **bigram** (`n=2`) recall for `candidate` vs. `reference`.

By hand: reference bigrams are (the,cat), (cat,sat), (sat,on), (on,the), (the,mat) — 5 total. The candidate reproduces 3 of them exactly ((the,cat), (cat,sat), (sat,on)); (on,mat) isn't a reference bigram, and (on,the)/(the,mat) are never produced by the candidate. So bigram recall should come out to $3/5 = 0.6$.

Implement `bigram_recall(candidate, reference)` below (it can just call `rouge_n_recall` you already wrote with the right `n`).

In [ ]:
def bigram_recall(candidate, reference):
    # TODO(you): return the ROUGE-2 (bigram) recall of candidate against reference
    return ...

result = bigram_recall(candidate, reference)
if result is not None:
    print('ROUGE-2 recall =', round(result, 4))
    assert abs(result - 0.6) < 1e-9
    print('Matches the hand-worked 3/5 = 0.6 — looks right.')

<details><summary>Solution</summary>

```python
def bigram_recall(candidate, reference):
    return rouge_n_recall(candidate, reference, 2)
```
`rouge_n_recall` was already generic in `n`; the only work is calling it with `n=2` instead of `n=1`. The clipped-overlap logic (counting shared bigrams, capped by how many times each appears in the reference) is identical to the unigram case.
</details>